# LangChain Structured Tool Reference

Developer-facing statements defined in `langchain_core.tools.structured`.

# `StructuredTool: BaseTool`

`StructuredTool` wraps a synchronous function, asynchronous coroutine, or both as a tool that accepts multiple named inputs.

## Fields

```python
description: str = "" # Description explaining when the tool should be used
args_schema: ArgsSchema # Pydantic model class or JSON schema describing tool inputs
func: Callable[..., Any] | None = None # Synchronous function executed by the tool
coroutine: Callable[..., Awaitable[Any]] | None = None # Asynchronous function executed by the tool
```

## Constructor

```python
StructuredTool(
    *,
    name: str, # Unique tool name
    description: str, # Tool purpose and usage description
    args_schema: ArgsSchema, # Input-validation schema
    func: Callable[..., Any] | None = None, # Synchronous function executed by the tool
    coroutine: Callable[..., Awaitable[Any]] | None = None, # Asynchronous function executed by the tool
    return_direct: bool = False, # Whether agent execution stops after this tool
    response_format: Literal[
        "content",
        "content_and_artifact",
    ] = "content", # Tool output format
    **kwargs: Any, # Additional BaseTool fields
) -> None # Initialize the tool
```

## Overridden Methods

### `ainvoke`

Executes the configured coroutine asynchronously.

When no coroutine is configured, it runs synchronous invocation in an executor.

### `_run`

Executes `func` synchronously.

Raises `NotImplementedError` when no synchronous function is configured.

### `_arun`

Executes `coroutine` asynchronously.

When no coroutine is configured, it delegates to the default asynchronous implementation of `BaseTool`.

## Class Method

### `from_function`

Creates a `StructuredTool` from a synchronous function, asynchronous coroutine, or both.

```python
StructuredTool.from_function(
    func: Callable[..., Any] | None = None, # Synchronous function executed by the tool
    coroutine: Callable[..., Awaitable[Any]] | None = None, # Asynchronous function executed by the tool
    name: str | None = None, # Tool name; defaults to the source function name
    description: str | None = None, # Tool description; defaults to the source function docstring
    return_direct: bool = False, # Whether agent execution stops after this tool
    args_schema: ArgsSchema | None = None, # Optional explicit input schema
    infer_schema: bool = True, # Whether the function signature generates the schema
    *,
    response_format: Literal[
        "content",
        "content_and_artifact",
    ] = "content", # Tool output format
    parse_docstring: bool = False, # Whether Google-style parameter descriptions are parsed
    error_on_invalid_docstring: bool = False, # Whether invalid parsed docstrings raise ValueError
    **kwargs: Any, # Additional StructuredTool fields
) -> StructuredTool # Return the generated tool
```

## Behaviour

- Accepts multiple named input arguments.
- Infers an input schema from the function signature when `infer_schema=True`.
- Excludes callback and Runnable configuration parameters from the inferred schema.
- Passes child callbacks when the wrapped callable accepts `callbacks`.
- Passes `RunnableConfig` when the wrapped callable declares a supported configuration parameter.
- Uses `(content, artifact)` output when `response_format="content_and_artifact"`.
- Raises `ValueError` when neither `func` nor `coroutine` is supplied.
- Raises `ValueError` when no description or usable docstring is available.
- Raises `TypeError` when `args_schema` is neither a Pydantic model class nor a dictionary.

from langchain_core.tools import StructuredTool # Import the multi-input tool class


def calculate_total(price: float, quantity: int, discount: float) -> float: # Define a function with multiple named inputs
    """Calculate the final price after applying a discount.""" # Provide the tool description
    subtotal: float = price * quantity # Calculate the price before discount
    return subtotal - (subtotal * discount / 100) # Return the final discounted amount


total_tool: StructuredTool = StructuredTool.from_function( # Create a StructuredTool from the function
    func=calculate_total, # Supply the synchronous function
    name="calculate_total", # Set the unique tool name
    description="Calculate the total price after applying a percentage discount", # Describe the tool
) # Finish creating the StructuredTool


result: float = total_tool.invoke( # Execute the tool with multiple named arguments
    {
        "price": 500.0, # Supply the price of one item
        "quantity": 3, # Supply the number of items
        "discount": 10.0, # Supply the discount percentage
    }
) # Finish invoking the tool


print("Tool name:", total_tool.name) # Display the tool name

print("Tool arguments:", total_tool.args) # Display the inferred argument schema

print("Final amount:", result) # Display the calculated result